# 生成数据集:3 个环境 x 2 个脚本 = 6 个 case

环境:`cartpole` / `mountain_car` / `pendulum`

脚本:
- `make_decoder_dataset` -> 生成训练 Decoder(WM)用的 (state, image) 数据对 + 真实 controller 轨迹
- `sampling` -> 生成转移数据集 (s,a,s') + 用指定 decoder（old / intensity / saliency）跑闭环验证轨迹

**运行前**:Jupyter kernel 切换成 `starv_shared`(`/home/tealab_shared/starv/env/starv_shared`)。

本文件放在 `verifiable_wm/notebooks/` 下,第一个 cell 会自动定位仓库根目录并切换过去。

In [5]:
import os
os.environ.setdefault("PYGLET_HEADLESS", "True")

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dynamic.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
assert (NOTEBOOK_DIR / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("repo root:", NOTEBOOK_DIR)

repo root: /home/UFAD/cjiang2/VWM/verifiable_wm


In [6]:
from utils import load_config
import make_decoder_dataset as mdd
import sampling as sp

def run_make_decoder_dataset(env_name, starv_only=False):
    """starv_only=True: 只重新生成 StarV 对齐的那套产物（starv_states.npz + 从它出发的
    real_trajectories.npz），不重写 decoder 训练用的 decoder_states.npz/图片/权重。等价命令行：
    python make_decoder_dataset.py config/make_decoder_dataset/<env>.json --starv-only"""
    config_path = NOTEBOOK_DIR / "config" / "make_decoder_dataset" / f"{env_name}.json"
    config = load_config(config_path)
    print(f"[Config] {config_path}")
    output_dir = mdd.generate_dataset(config, starv_only=starv_only)
    print(f"[Done] decoder dataset saved to {output_dir}")
    return output_dir

def run_sampling(env_name, decoder_variant=None):
    """decoder_variant: 闭环轨迹用哪个 decoder 跑（old / intensity / saliency）。
    None 时用 config 里 decoder.variant 的值；输出文件为 dwm_trajectories_<variant>.npz。"""
    config_path = NOTEBOOK_DIR / "config" / "sampling" / f"{env_name}.json"
    config = load_config(config_path)
    if decoder_variant is not None:
        config["decoder"]["variant"] = decoder_variant
    print(f"[Config] {config_path}")
    output_dir = sp.generate_dataset(config)
    print(f"[Done] transition dataset saved to {output_dir}")
    return output_dir

## Case 1: CartPole - make_decoder_dataset

完整生成，写入 `datasets/cartpole/data/dataset_v1/` 下四个文件：

- `decoder_states.npz`：decoder 训练数据（`{split}_states` + 渲染的 `{split}_images`）
- `metadata.json`：本次生成用的 config 快照
- `starv_states.npz`：从 StarV grid 采样的完整初始 state
- `real_trajectories.npz`：从这些 state 出发、用真实 renderer 跑的闭环轨迹

In [3]:
run_make_decoder_dataset("cartpole")

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/make_decoder_dataset/cartpole.json
[Load] Controller=/home/tealab_shared/starv/weights/cartpole/controller_cp.pth
[Dynamic] CartPole


/home/tealab_shared/starv/env/starv_shared/lib/python3.9/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


[Render] train: states=(1600, 4), images=(1600, 1, 96, 96)
[Render] val: states=(400, 4), images=(400, 1, 96, 96)
[Render] test: states=(400, 4), images=(400, 1, 96, 96)
[Saved] starv states=datasets/cartpole/data/dataset_v1/starv_states.npz
[Trajectory] train: traj=(1600, 31, 4), actions=(1600, 30, 1)
[Trajectory] val: traj=(400, 31, 4), actions=(400, 30, 1)
[Trajectory] test: traj=(400, 31, 4), actions=(400, 30, 1)
[Done] decoder dataset saved to datasets/cartpole/data/dataset_v1


PosixPath('datasets/cartpole/data/dataset_v1')

## Case 1b: CartPole - starv-only

只生成两个文件，不动 decoder 训练数据：

- `starv_states.npz`：从 StarV grid 采样的完整初始 state
- `real_trajectories.npz`：从这些 state 出发、用真实 renderer 跑的闭环轨迹

In [7]:
run_make_decoder_dataset("cartpole", starv_only=True)

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/make_decoder_dataset/cartpole.json


TypeError: generate_dataset() got an unexpected keyword argument 'starv_only'

## Case 2: CartPole - sampling

读取 `starv_states.npz` 作为初始 state（所以必须先跑 Case 1 或 1b），生成：

- `transition_dataset.npz`：真实 renderer 下的单步 `(s, a, s')` transition
- `dwm_trajectories_<variant>.npz`：用指定 decoder 替代 renderer 的闭环轨迹，起点和 real 完全相同

In [ ]:
# 闭环轨迹用哪个 decoder 跑：三选一，注释掉另外两个
# 输出互不覆盖，分别存为 dwm_trajectories_old / _intensity / _saliency.npz

run_sampling("cartpole", decoder_variant="old")          # 一代老 decoder
# run_sampling("cartpole", decoder_variant="intensity")  # 复现论文 baseline
# run_sampling("cartpole", decoder_variant="saliency")   # 我们的 saliency 方法

## Case 3: MountainCar - make_decoder_dataset

产物同 Case 1，目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [5]:
run_make_decoder_dataset("mountain_car")

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/make_decoder_dataset/mountain_car.json
[Load] Controller=/home/tealab_shared/starv/weights/mountain_car/controller_mc.pth
[Dynamic] MountainCar
[Render] train: states=(1600, 2), images=(1600, 1, 96, 96)
[Trajectory] train: traj=(1600, 31, 2), actions=(1600, 30, 1)
[Render] val: states=(400, 2), images=(400, 1, 96, 96)
[Trajectory] val: traj=(400, 31, 2), actions=(400, 30, 1)
[Render] test: states=(400, 2), images=(400, 1, 96, 96)
[Trajectory] test: traj=(400, 31, 2), actions=(400, 30, 1)
[Done] decoder dataset saved to datasets/mountain_car/data/dataset_v1


PosixPath('datasets/mountain_car/data/dataset_v1')

## Case 3b: MountainCar - starv-only

产物同 Case 1b（`starv_states.npz` + `real_trajectories.npz`），目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("mountain_car", starv_only=True)

## Case 4: MountainCar - sampling

产物同 Case 2，目录换成 `datasets/mountain_car/data/dataset_v1/`。

In [6]:
run_sampling("mountain_car")

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/sampling/mountain_car.json
[Load] Controller=/home/tealab_shared/starv/weights/mountain_car/controller_mc.pth
[Load] Decoder=/home/tealab_shared/starv/weights/mountain_car/decoder_mc.pth
[Dynamic] MountainCar
[Rollout] train: states=(48000, 2), actions=(48000, 1), next_states=(48000, 2)
[DWM Trajectory] train: traj=(1600, 31, 2), actions=(1600, 30, 1)
[Rollout] val: states=(12000, 2), actions=(12000, 1), next_states=(12000, 2)
[DWM Trajectory] val: traj=(400, 31, 2), actions=(400, 30, 1)
[Rollout] test: states=(12000, 2), actions=(12000, 1), next_states=(12000, 2)
[DWM Trajectory] test: traj=(400, 31, 2), actions=(400, 30, 1)
[Done] transition dataset saved to datasets/mountain_car/data/dataset_v1


PosixPath('datasets/mountain_car/data/dataset_v1')

## Case 5: Pendulum - make_decoder_dataset

产物同 Case 1，目录换成 `datasets/pendulum/data/dataset_v1/`。

In [7]:
run_make_decoder_dataset("pendulum")

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/make_decoder_dataset/pendulum.json
[Load] Controller=/home/tealab_shared/starv/weights/pendulum/controller_pen.pth
[Dynamic] Pendulum
[Render] train: states=(1600, 2), images=(1600, 1, 96, 96)
[Trajectory] train: traj=(1600, 31, 2), actions=(1600, 30, 1)
[Render] val: states=(400, 2), images=(400, 1, 96, 96)
[Trajectory] val: traj=(400, 31, 2), actions=(400, 30, 1)
[Render] test: states=(400, 2), images=(400, 1, 96, 96)
[Trajectory] test: traj=(400, 31, 2), actions=(400, 30, 1)
[Done] decoder dataset saved to datasets/pendulum/data/dataset_v1


PosixPath('datasets/pendulum/data/dataset_v1')

## Case 5b: Pendulum - starv-only

产物同 Case 1b（`starv_states.npz` + `real_trajectories.npz`），目录换成 `datasets/pendulum/data/dataset_v1/`。

In [ ]:
run_make_decoder_dataset("pendulum", starv_only=True)

## Case 6: Pendulum - sampling

产物同 Case 2，目录换成 `datasets/pendulum/data/dataset_v1/`。

In [8]:
run_sampling("pendulum")

[Config] /home/UFAD/cjiang2/VWM/verifiable_wm/config/sampling/pendulum.json
[Load] Controller=/home/tealab_shared/starv/weights/pendulum/controller_pen.pth
[Load] Decoder=/home/tealab_shared/starv/weights/pendulum/decoder_pen.pth
[Dynamic] Pendulum
[Rollout] train: states=(48000, 2), actions=(48000, 1), next_states=(48000, 2)
[DWM Trajectory] train: traj=(1600, 31, 2), actions=(1600, 30, 1)
[Rollout] val: states=(12000, 2), actions=(12000, 1), next_states=(12000, 2)
[DWM Trajectory] val: traj=(400, 31, 2), actions=(400, 30, 1)
[Rollout] test: states=(12000, 2), actions=(12000, 1), next_states=(12000, 2)
[DWM Trajectory] test: traj=(400, 31, 2), actions=(400, 30, 1)
[Done] transition dataset saved to datasets/pendulum/data/dataset_v1


PosixPath('datasets/pendulum/data/dataset_v1')

## 汇总:检查所有生成的文件

In [9]:
for env_name in ("cartpole", "mountain_car", "pendulum"):
    data_dir = NOTEBOOK_DIR / "datasets" / env_name / "data" / "dataset_v1"
    print(f"--- {env_name} ({data_dir}) ---")
    if data_dir.exists():
        for f in sorted(data_dir.iterdir()):
            print(f"  {f.name}: {f.stat().st_size} bytes")
    else:
        print("  (尚未生成)")

--- cartpole (/home/UFAD/cjiang2/VWM/verifiable_wm/datasets/cartpole/data/dataset_v1) ---
  dwm_trajectories.npz: 1357876 bytes
  metadata.json: 1043 bytes
  real_trajectories.npz: 1363261 bytes
  states.npz: 521804 bytes
  transition_dataset.npz: 2405578 bytes
--- mountain_car (/home/UFAD/cjiang2/VWM/verifiable_wm/datasets/mountain_car/data/dataset_v1) ---
  dwm_trajectories.npz: 818688 bytes
  metadata.json: 847 bytes
  real_trajectories.npz: 707754 bytes
  states.npz: 1705369 bytes
  transition_dataset.npz: 1240319 bytes
--- pendulum (/home/UFAD/cjiang2/VWM/verifiable_wm/datasets/pendulum/data/dataset_v1) ---
  dwm_trajectories.npz: 809021 bytes
  metadata.json: 875 bytes
  real_trajectories.npz: 695931 bytes
  states.npz: 506401 bytes
  transition_dataset.npz: 1211186 bytes
